# Model Validation Before Registry

## Topic Goal

This notebook demonstrates how to implement a **model validation gate** before registering a model in MLflow Model Registry.

- The model is trained and evaluated.
- Validation thresholds are defined.
- The model is logged with signature and input example in the **same MLflow run**.
- The model is registered **only if** the validation gate passes.
- If the model fails the validation gate, it is logged but **not registered**.

This is a realistic production MLOps pattern because not every trained model should be promoted or registered.


## 1. Import Libraries

We import MLflow, pandas, scikit-learn, and utilities required for model training, evaluation, signature generation, and registry operations.


In [1]:
# Import os to create folders and clean MLflow project environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for experiment tracking, model logging, and model registry.
import mlflow

# Import MLflow scikit-learn integration for logging sklearn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input and output schema.
from mlflow.models.signature import infer_signature

# Import pandas for reading and processing CSV datasets.
import pandas as pd

# Import numpy for numerical operations.
import numpy as np

# Import train_test_split for splitting dataset.
from sklearn.model_selection import train_test_split

# Import ColumnTransformer for applying different preprocessing to different column types.
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder for categorical columns.
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler for numerical columns.
from sklearn.preprocessing import StandardScaler

# Import Pipeline to combine preprocessing and model into one deployable object.
from sklearn.pipeline import Pipeline

# Import RandomForestClassifier for churn prediction.
from sklearn.ensemble import RandomForestClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## 2. Configure MLflow

This section configures the notebook to use a local `mlruns` folder.

For Windows classroom demos, this approach is simple and reliable.


In [2]:
# Define experiment name for this notebook.
EXPERIMENT_NAME = "advanced_06_model_validation_gate"

# Define run name for the same-run use case.
RUN_NAME = "06_model_validation_gate_same_run_usecase"

# Remove inherited MLflow Project variables to avoid experiment ID issues.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

# Remove inherited run ID if any old run context exists.
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local MLflow tracking folder.
os.makedirs("mlruns", exist_ok=True)

# Create artifacts folder for any local files.
os.makedirs("artifacts", exist_ok=True)

# Create outputs folder for generated files.
os.makedirs("outputs", exist_ok=True)

# Configure MLflow to use local file-based tracking.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset path.
DATA_PATH = "datasets/customer_churn_500.csv"

# Set MLflow experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print configuration details.
print("Experiment:", EXPERIMENT_NAME)
print("Run name:", RUN_NAME)
print("Registered model name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Dataset path:", DATA_PATH)


2026/05/14 13:24:53 INFO mlflow.tracking.fluent: Experiment with name 'advanced_06_model_validation_gate' does not exist. Creating a new experiment.


Experiment: advanced_06_model_validation_gate
Run name: 06_model_validation_gate_same_run_usecase
Registered model name: advanced_06_model_validation_gate_Model
Tracking URI: file:./mlruns
Dataset path: datasets/customer_churn_500.csv


## 3. Load and Prepare Dataset

We use the customer churn dataset.

The target column is `churn`.

The model will predict whether a customer is likely to churn.


In [3]:
# Load the customer churn dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define the target column.
target_column = "churn"

# Create feature matrix by dropping customer ID and target column.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Print feature groups for explanation.
print("Categorical columns:", categorical_columns)
print("Numerical columns:", numerical_columns)

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Standardize numerical columns.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical columns.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Print shape details.
print("Dataset shape:", df.shape)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


,customer_id,age,gender,region,plan_type,tenure_months,monthly_charges,support_tickets,data_usage_gb,contract_type,churn
0,CUST1060001,45,Male,South,Standard,16,2782.25,1,145.76,One year,0
1,CUST1060002,65,Male,South,Premium,22,2648.21,1,98.84,Month-to-month,0
2,CUST1060003,40,Male,East,Premium,50,1637.93,2,141.16,Month-to-month,0
3,CUST1060004,22,Male,North,Basic,62,1801.85,4,108.28,Month-to-month,1
4,CUST1060005,30,Female,West,Standard,36,2370.39,2,38.27,One year,0


Categorical columns: ['gender', 'region', 'plan_type', 'contract_type']
Numerical columns: ['age', 'tenure_months', 'monthly_charges', 'support_tickets', 'data_usage_gb']
Dataset shape: (500, 11)
Training rows: 375
Testing rows: 125


## 4. Define Validation Gate

A validation gate is a production rule that decides whether a model is good enough for registration.

In this example:

- The model must have F1-score greater than or equal to `minimum_f1`.
- The model must have recall greater than or equal to `minimum_recall`.

Only then will the model be registered.


In [4]:
# Define minimum required F1-score.
minimum_f1 = 0.55

# Define minimum required recall.
minimum_recall = 0.50

# Create explanation artifact for the validation gate.
with open("outputs/topic_artifact.txt", "w") as file:
    # Write purpose of validation gate.
    file.write("This notebook demonstrates model validation before registry.\n")

    # Write F1 threshold.
    file.write(f"Minimum F1 threshold: {minimum_f1}\n")

    # Write recall threshold.
    file.write(f"Minimum recall threshold: {minimum_recall}\n")

# Print validation thresholds.
print("Minimum F1-score required:", minimum_f1)
print("Minimum recall required:", minimum_recall)


Minimum F1-score required: 0.55
Minimum recall required: 0.5


## 5. Train and Evaluate Model

We train a Random Forest model using a complete scikit-learn pipeline.

The pipeline contains both preprocessing and model training.

This is important because the logged MLflow model should accept raw input columns.


In [5]:
# Create Random Forest classifier.
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=6,
    random_state=42
)

# Create complete pipeline with preprocessing and model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train the Random Forest classifier.
    ("model", model),
])

# Train the pipeline.
pipeline.fit(X_train, y_train)

# Generate predictions on the test dataset.
predictions = pipeline.predict(X_test)

# Calculate accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Print model metrics.
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Create and print classification report.
report = classification_report(y_test, predictions, zero_division=0)
print(report)


Accuracy: 0.632
Precision: 0.6153846153846154
Recall: 0.16326530612244897
F1-score: 0.25806451612903225
              precision    recall  f1-score   support

           0       0.63      0.93      0.76        76
           1       0.62      0.16      0.26        49

    accuracy                           0.63       125
   macro avg       0.62      0.55      0.51       125
weighted avg       0.63      0.63      0.56       125



## 6. Check Validation Gate

This is the most important part of this notebook.

The model should be registered only if it passes the validation rule.


In [6]:
# Check whether model passes the validation gate.
passed_gate = (f1 >= minimum_f1) and (recall >= minimum_recall)

# Create validation status text.
validation_status = "passed" if passed_gate else "failed"

# Print validation status.
print("Validation status:", validation_status)

# Print reason clearly for students.
if passed_gate:
    # Message when model passes.
    print("Model passed the validation gate and can be registered.")
else:
    # Message when model fails.
    print("Model failed the validation gate and will not be registered.")
    print("F1-score:", f1, "Required:", minimum_f1)
    print("Recall:", recall, "Required:", minimum_recall)


Validation status: failed
Model failed the validation gate and will not be registered.
F1-score: 0.25806451612903225 Required: 0.55
Recall: 0.16326530612244897 Required: 0.5


## 7. Infer Signature, Log Model, and Register Conditionally from the Same Run

This section keeps the flow production-friendly:

1. Create input example.
2. Infer model signature.
3. Start one MLflow run.
4. Log parameters, thresholds, metrics, validation status, artifacts, and model.
5. Create `model_uri` from the same run.
6. Register only if the validation gate passes.

This avoids creating a separate registry/signature run.


In [7]:
# Create input example for signature.
input_example = X_test.head(5)

# Generate sample predictions for the input example.
sample_predictions = pipeline.predict(input_example)

# Infer model signature from input example and sample predictions.
signature = infer_signature(input_example, sample_predictions)

# Start the SAME MLflow run for validation, metrics, signature, model, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic/experiment name.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log validation gate thresholds.
    mlflow.log_param("minimum_f1", minimum_f1)
    mlflow.log_param("minimum_recall", minimum_recall)

    # Log validation result as parameter.
    mlflow.log_param("passed_validation_gate", passed_gate)

    # Log accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log precision.
    mlflow.log_metric("precision", precision)

    # Log recall.
    mlflow.log_metric("recall", recall)

    # Log F1-score.
    mlflow.log_metric("f1_score", f1)

    # Set validation status as tag.
    mlflow.set_tag("validation_status", validation_status)

    # Set model registration decision as tag.
    mlflow.set_tag("registration_decision", "register" if passed_gate else "do_not_register")

    # Save classification report locally.
    with open("outputs/classification_report.txt", "w") as file:
        file.write(report)

    # Log classification report artifact.
    mlflow.log_artifact("outputs/classification_report.txt")

    # Log validation gate explanation artifact.
    if os.path.exists("outputs/topic_artifact.txt"):
        mlflow.log_artifact("outputs/topic_artifact.txt")

    # Log model WITH signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Print same-run model URI.
print("Same-run model URI:", model_uri)


Same-run model URI: runs:/b09aa6254cee43dda8e626ed7293eaea/model


## 8. Register Model Only If Validation Gate Passes

This is the corrected registry logic.

The model is **not registered unconditionally**.

It is registered only when `passed_gate == True`.


In [8]:
# Register model only if validation gate passed.
if passed_gate:
    # Register model using the same-run model URI.
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name=REGISTERED_MODEL_NAME
    )

    # Print registration success.
    print("Validation gate passed.")
    print("Model registered successfully.")
    print("Registered model name:", registered_model.name)
    print("Registered model version:", registered_model.version)

else:
    # Do not register the model when validation fails.
    registered_model = None

    # Print registration skipped message.
    print("Validation gate failed.")
    print("Model was logged for traceability but was NOT registered.")
    print("Model URI:", model_uri)
    print("F1-score:", f1, "| Required F1:", minimum_f1)
    print("Recall:", recall, "| Required recall:", minimum_recall)


Validation gate failed.
Model was logged for traceability but was NOT registered.
Model URI: runs:/b09aa6254cee43dda8e626ed7293eaea/model
F1-score: 0.25806451612903225 | Required F1: 0.55
Recall: 0.16326530612244897 | Required recall: 0.5


## 9. Trainer Notes



- In production, not every trained model should enter the registry.
- A model should pass predefined quality rules before registration.
- This notebook logs the model even if it fails, because failed runs are also useful for audit and comparison.
- Registration happens only when the model passes the gate.
- The model signature, input example, model artifact, metrics, validation status, and model URI all belong to the same MLflow run.
- This is a clean MLOps pattern for automated training pipelines.
